## **Notebook Table of Contents**

## Notebook Table of Contents

1. [Initial Setup and Data Loading](#1-initial-setup-and-data-loading)

2. [Categorical Variable Exploration and Cleaning](#2-categorical-variable-exploration-and-cleaning)

3. [Numerical Variable Exploration and Cleaning](#3-numerical-variable-exploration-and-cleaning)

4. [Missing Values Exploration](#4-missing-values-exploration)

5. [Data Split and Preprocessing Pipeline](#5-data-split-and-preprocessing-pipeline)

6. [Feature Selection](#6-feature-selection)

7. [Model Selection](#7-model-selection)

8. [Extra Trees — Final Hyperparameter Tuning](#8-extra-trees--final-hyperparameter-tuning)

9. [Model Blending](#9-model-blending)

10. [Analytics Interface for New-Data Inference](#10-analytics-interface-for-new-data-inference)



### Group Member Contribution — Group 57

**20211548 – Carlos Amorim**  
Exploratory data analysis and data cleaning, ensuring data quality and consistency for modeling.  
Estimated contribution: **25%**

**20250468 – Rodrigo Andrade**  
Design and implementation of preprocessing strategies, supporting consistent data preparation and model training.  
Estimated contribution: **25%**

**20250468 – Francisco Cerdeira**  
Development of the first prototype for preprocessing and feature selection, supporting model evaluation.  
Estimated contribution: **25%**

**2014553 – António Varela**  
Model selection, hyperparameter tuning, ensemble definition, and development of the analytics interface and inference pipeline.  
Estimated contribution: **25%**


### Abstract

The price of a used car is influenced by multiple factors such as age, mileage, brand, fuel type, and technical characteristics, making accurate price estimation a complex task. The objective of this project is to develop a consistent model assessment strategy that allows the creation and comparison of different candidate regression models in order to identify the most generalizable one for predicting car prices.

The work started with an exploratory analysis of the dataset to understand feature distributions and identify data quality issues. A major challenge during data preparation was the cleaning of categorical variables, particularly brand and model names, which contained numerous inconsistencies and misspellings. To address this, a hybrid fuzzy string matching approach was applied, combining Euclidean distance and Levenshtein distance to standardise equivalent values. Missing data was handled using appropriate imputation methods, while numerical features were scaled and categorical features were encoded. The dataset was split into training and test sets, with all model training, comparison, and hyperparameter tuning performed exclusively on the training data.

In total, six different regression models were developed and evaluated using a consistent assessment strategy, enabling a fair comparison of their performance and generalisation capability. The best results, in terms of Mean Absolute Error (MAE), were achieved using a blended model combining Extra Trees, Random Forest, and Histogram Gradient Boosting. In addition, an analytics interface was created using Gradio, allowing users to input new vehicle data and instantly receive a predicted price from the trained model.

Future work may focus on feature engineering, expanding the dataset, and improving the user interface to make it more user-friendly and robust.


In [ ]:
"""
README — Cars4You Machine Learning Pipeline
===========================================

Computational Requirements
--------------------------
This notebook contains the complete Cars4You machine learning pipeline, covering
all stages from data preparation to model inference and user interaction.

The pipeline includes computationally intensive steps such as high-dimensional
one-hot encoding, ensemble tree models, and final model blending. For this reason,
the full pipeline was executed using Google Colab with a high-memory runtime.

Minimum recommended resources to execute the full pipeline:
- RAM: at least 52 MB
- CPU: standard multi-core environment
- Execution time: several minutes, depending on the runtime configuration

Running the entire pipeline on machines with lower memory availability may result
in kernel interruptions or incomplete execution.

--------------------------------------------------

Input Data Requirements
-----------------------
The execution of this pipeline requires the availability of the original dataset
files provided with the project:

- `train.csv`: loaded in **Section 1**, used for data exploration, preprocessing,
  model training, and validation.
- `test.csv`: loaded in **Section 8**, used exclusively for final inference and
  generation of test predictions.

Both files must be accessible in the runtime environment before executing the
corresponding sections of the notebook.

--------------------------------------------------

Project Requirements and Compliance
-----------------------------------
This project was designed to meet the specified requirements of the assignment.
In particular, it explicitly satisfies requirement (c):

"Create an analytics interface that returns a prediction when new input data is
provided."

This requirement is fulfilled through:
- a inference function that applies the original, leakage-safe
  preprocessing pipeline to new, unseen data
- consistent reuse of trained artifacts, including encoders, imputation values,
  and the final feature space
- an interactive analytics interface that allows users to input new car data and
  obtain price predictions

The analytics interface provides predictions from multiple trained models as well
as a weighted ensemble prediction, together with a visual comparison of model
outputs.

--------------------------------------------------

Note to the Reader
------------------
This notebook reflects a complete end-to-end workflow, from raw data processing to
a user-facing analytics interface. The pipeline is intentionally structured to
clearly separate training, inference, and interaction layers, ensuring clarity,
reproducibility, and ease of evaluation.

This README describes the current state of the project and may be updated in future
iterations to reflect optimizations or additional deployment considerations.
"""


# **1. Initial Setup and Data Loading**

In [ ]:
# =========================================
#  1) Instalação de pacotes necessários
# =========================================
!pip install rapidfuzz


In [ ]:
# =========================================
# 2) Importações — Organização por tema
# =========================================

# ---- Manipulação de dados
import pandas as pd
import numpy as np

# ---- Similaridade / fuzzy matching
from rapidfuzz import process, fuzz, distance

# ---- Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# ---- Pré-processamento
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    StandardScaler,
    LabelEncoder
)

# ---- Estatística
import scipy.stats as stats
from scipy.stats import chi2_contingency, f_oneway

# ---- Feature Selection
from sklearn.feature_selection import (
    mutual_info_classif,
    mutual_info_regression,
    RFE
)

# ---- Modelos
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

# ---- Métricas
from sklearn.metrics import (
    mean_squared_error,
    make_scorer,
    r2_score,
    mean_absolute_error
)

# ---- Validação e splits
from sklearn.model_selection import (
    train_test_split,
    KFold,
    RepeatedKFold
)

# ---- Reprodutibilidade
import random

# ---- Upload (Google Colab)
from google.colab import files


# =========================================
#  3) Fixar Random Seed
# =========================================
RANDOM_SEED = 1907
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
# =========================================
#  4) Upload do dataset via Colab
# =========================================
uploaded = files.upload()  # O utilizador escolhe o ficheiro manualmente


# =========================================
# 5) Carregar e inspecionar os dados
# =========================================
df = pd.read_csv("train.csv")  # Nome do ficheiro depois do upload
print("\nArquivo carregado:", "train.csv")

# =========================================
#  TOY MODE (opcional)
# =========================================
USE_TOY = False   # <-- Mude para False para usar o dataset completo

if USE_TOY:
    print("\n TOY MODE ATIVO — utilizando apenas 1000 linhas para testes.")
    df = df.sample(n=1000, random_state=42).reset_index(drop=True)
else:
    print("\nFULL MODE — utilizando todas as linhas do train.csv.")

# Mostrar forma do dataset usado
print("\nDataset carregado com shape:", df.shape)

# Guardar cópia original para referência
og_df = df.copy()

# Mostrar primeiras linhas
display(df)

# Estrutura do DataFrame
print("\n=== df.info() ===")
df.info()

# Estatísticas descritivas
print("\n=== df.describe() ===")
display(df.describe())



In [ ]:
df["carID"].duplicated().any()

In [ ]:

def carID_como_index(df):
    df = df.copy()
    df = df.set_index("carID")
    return df

df=carID_como_index(df)
display(df)

# **2. Categorical Variable Exploration and Cleaning**

## **2.1 Brand Feature Exploration and Cleaning**

In [ ]:
df["Brand"].unique()

In [ ]:
df["Brand"].value_counts()

In [ ]:
df[df["Brand"].isna()]

In [ ]:
def HELPER_marca_correta(marca, marcas_dict, threshold):
    if not isinstance(marca, str) or marca.strip() == "": #Check type and emptiness & Lowercase and strip:
        return marca

    result = process.extractOne(
        marca.lower().strip(),
        marcas_dict.keys(),
        scorer=fuzz.token_sort_ratio
    )

    #result = (match, score, index)
    #ex: match = toyota, score = 94.5, index = 1 (qual das marcas no dicionário é mais semelhante)

    if result is None:
        return marca

    match_lower, score, _ = result
    return marcas_dict[match_lower] if score >= threshold else marca #Score check



In [ ]:
def fuzzy_marcas(df, threshold=50):
    df=df.copy()

    valid_brands = ["VW", "Toyota", "Audi", "Ford", "BMW", "Opel", "Skoda", "Mercedes", "Hyundai"]
    brands_dict = {BRAND.lower(): BRAND for BRAND in valid_brands}

    df["Brand"] = df["Brand"].apply(lambda x: HELPER_marca_correta(x, brands_dict, threshold))
    return df

In [ ]:
df=fuzzy_marcas(df)
df["Brand"].unique()

In [ ]:
df["Brand"].value_counts()

## **2.2 MODEL Feature Exploration and Cleaning**

In [ ]:
df["model"].unique()

In [ ]:
len(df["model"].unique())

In [ ]:
# group models by brand
grouped = df.groupby("Brand")["model"].value_counts()

display(grouped["Hyundai"])


In [ ]:
"""Optained through meticulous ChatGPT prompt engineering"""
valid_models = {
    "Audi": [
        "A1", "A2", "A3", "A4", "A5", "A6", "A7", "A8",
        "Q2", "Q3", "Q5", "Q7", "Q8",
        "TT", "T", "R8",
        "S3", "S4", "S5", "S8",
        "RS3", "RS4", "RS5", "RS6",
        "SQ5", "SQ7"
    ],

    "Ford": [
        "FOCUS", "FIESTA", "MONDEO", "KA", "KA+", "FUSION",
        "KUGA", "ECOSPORT", "EDGE", "PUMA",
        "CMAX", "BMAX", "SMAX", "GALAXY",
        "GRANDCMAX", "TOURNEOCONNECT", "GRANDTOURNEOCONNECT", "TOURNEOCUSTOM",
        "MUSTANG", "RANGER", "ESCORT", "STREETKA"
    ],

    "Mercedes": [
        "ACLASS", "BCLASS", "CCLASS", "ECLASS", "SCLASS",
        "GLA", "GLB", "GLC", "GLE", "GLS", "GCLASS", "GLCLASS", "MCLASS",
        "CLA", "CLS", "SL", "SLK", "CLK",
        "VCLASS", "XCLASS", "CLC"
    ],

    "VW": [
        "GOLF", "POLO", "PASSAT", "JETTA", "ARTEON", "SCIROCCO", "BEETLE",
        "UP", "GOL", "FOX",
        "TIGUAN", "TIGUANALLSPACE", "TROC", "TCROSS", "TOUAREG",
        "TOURAN", "SHARAN", "CADDY", "CADDYMAXI", "CADDYMAXILIFE",
        "CARAVELLE", "CALIFORNIA", "SHUTTLE",
        "AMAROK", "GOLFSV", "CC"
    ],

    "Opel": [
        "CORSA", "ASTRA", "INSIGNIA", "VECTRA",
        "MOKKA", "MOKKAX", "CROSSLAND", "CROSSLANDX",
        "GRANDLAND", "GRANDLANDX", "ANTARA",
        "ZAFIRA", "ZAFIRATOURER", "MERIVA", "COMBOLIFE", "VIVARO",
        "ADAM", "AGILA", "VIVA",
        "TIGRA", "GTC", "CASCADA", "AMPERA"
    ],

    "BMW": [
        "1SERIES", "2SERIES", "3SERIES", "4SERIES",
        "5SERIES", "6SERIES", "7SERIES", "8SERIES",
        "X1", "X2", "X3", "X4", "X5", "X6", "X7",
        "M2", "M3", "M4", "M5", "M6",
        "Z3", "Z4",
        "I3", "I4", "I8"
    ],

    "Toyota": [
        "YARIS", "AYGO", "AURIS", "COROLLA", "AVENSIS", "CAMRY", "PRIUS",
        "CHR", "RAV4", "LANDCRUISER", "URBANCRUISER",
        "VERSO", "VERSOS", "PROACEVERSO",
        "HILUX", "GT86", "SUPRA", "IQ"
    ],

    "Skoda": [
        "FABIA", "OCTAVIA", "SUPERB", "RAPID", "SCALA",
        "KODIAQ", "KAROQ", "KAMIQ", "YETI", "YETIOUTDOOR",
        "CITIGO", "ROOMSTER"
    ],

    "Hyundai": [
        "I10", "I20", "I30", "I40", "ACCENT", "GETZ",
        "KONA", "TUCSON", "SANTAFE", "IX20", "IX35",
        "I800", "IONIQ", "VELOSTER", "TERRACAN"
    ]
}


In [ ]:
def HELPER_normalize_models(df):
    df = df.copy()

    df["model"] = (
        df["model"]
        .astype(str)
        .str.upper()
        .str.replace("-", "", regex=False)       # remove hyphens
        .str.replace(r"\s+", "", regex=True)     # remove all whitespace
        .replace(["", "NAN", "NONE"], None)
    )
    return df

def HELPER_hybrid_scorer(a, b, **kwargs):
    #hibrido entre o método tokenizer e o levenshtein
    lev = distance.Levenshtein.normalized_similarity(a, b)
    token = fuzz.token_sort_ratio(a, b) / 100
    return (0.7 * lev + 0.3 * token) * 100

def HELPER_modelo_correto(model, brand, valid_models_dict, threshold):
    if not model or model.strip() == "":
        return None

    if not brand:
        return model

    # Skip unknown brands
    if brand not in valid_models_dict:
        return model

    valid_list = valid_models_dict[brand]

    result = process.extractOne(model, valid_list, scorer=HELPER_hybrid_scorer)

    if result is None:
        return model

    model_name, score, _ = result
    return model_name if score >= threshold else None

def fuzzy_modelos(df, valid_models_dict, threshold=30):
    df = HELPER_normalize_models(df)

    df["model"] = df.apply(
        lambda row: HELPER_modelo_correto(row["model"], row["Brand"], valid_models_dict, threshold),
        axis=1
    )
    return df

In [ ]:
len(df["model"].unique())
df[df["model"].isna()]

In [ ]:
unique_models_per_brand = df.groupby("Brand")["model"].nunique()
print(unique_models_per_brand)

In [ ]:
df=fuzzy_modelos(df, valid_models)
display(df[df["model"].isna()])

In [ ]:
unique_models_per_brand = df.groupby("Brand")["model"].nunique()
print(unique_models_per_brand)

## 2.3 **MODELS Brand Extraction**

In [ ]:
def inferir_marca_com_modelo(df, valid_models_dict):
    df = df.copy()

 # Build a mapping of models → their most frequent (mode) brand in the dataset
    model_to_brand = (
        df[df["Brand"].notna()]
        .groupby("model")["Brand"]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        .to_dict()
    )

    df["Brand"] = df.apply(
    lambda row: (
        row["Brand"]
        if pd.notna(row["Brand"])
        else model_to_brand.get(row["model"], None)
    ),
    axis=1
    )

    return df


In [ ]:
df[df["Brand"].isna()]

In [ ]:
df=inferir_marca_com_modelo(df, valid_models)
df[df["Brand"].isna()]

In [ ]:
df=fuzzy_modelos(df, valid_models)
display(df[df["model"].isna()])

In [ ]:
df

In [ ]:
missing_model_count = df["model"].isna().sum()
print("Number of cars without a model:", missing_model_count)


In [ ]:
# Substituir valores ausentes (NaN) em 'model' por 'Unknown'
df["model"] = df["model"].fillna("Unknown")


In [ ]:
display(df[df["model"].isna()])

In [ ]:
display(df[df["model"] == "Unknown"])


## **2.4 Transmission Exploration and Cleaning**

In [ ]:
df["transmission"].unique()


In [ ]:
df[df["transmission"].isna()]

In [ ]:
def HELPER_normalize_transmission(df):
    df = df.copy()
    df["transmission"] = (
        df["transmission"]
        .astype(str)
        .str.strip()
        .str.upper()
        .replace(["", "NAN", "NONE"], np.nan)
    )
    return df

In [ ]:
def HELPER_transmissao_correta(transm, valid_list, threshold):
    if pd.isna(transm):
        return np.nan

    if transm in valid_list:
        return transm

    result = process.extractOne(transm, valid_list, scorer=fuzz.token_sort_ratio)
    if result is None:
        return np.nan

    match_name, score, _ = result
    return match_name if score >= threshold else np.nan

def fuzzy_transmissao(df, threshold=60):
    df = HELPER_normalize_transmission(df)
    valid_list = ["MANUAL", "AUTOMATIC", "SEMI-AUTO", "OTHER", "UNKNOWN"]

    df["transmission"] = df["transmission"].apply(
        lambda x: HELPER_transmissao_correta(x, valid_list, threshold)
    )
    return df

In [ ]:
df = fuzzy_transmissao(df)
df[df["transmission"].isna()]

In [ ]:
df["transmission"].unique()

## **2.5 Fuel Type analysis**

In [ ]:
df["fuelType"].unique()

In [ ]:
df[df["fuelType"].isna()]

In [ ]:
df["fuelType"].value_counts()

In [ ]:
def HELPER_normalize_fueltype(df):
    df = df.copy()
    df["fuelType"] = (
        df["fuelType"]
        .astype(str)
        .str.strip()
        .str.upper()
        .replace(["", "NAN", "NONE"], np.nan)
    )
    return df

In [ ]:
def HELPER_fuel_correto(fuel, valid_list, threshold):
    if pd.isna(fuel):
        return np.nan

    if fuel in valid_list:
        return fuel

    result = process.extractOne(fuel, valid_list, scorer=fuzz.token_sort_ratio)
    if result is None:
        return np.nan

    match_name, score, _ = result
    return match_name if score >= threshold else np.nan

def fuzzy_fuel(df, threshold=60):
    df = HELPER_normalize_fueltype(df)
    valid_list = ["PETROL", "DIESEL", "HYBRID", "OTHER", "ELECTRIC"]

    df["fuelType"] = df["fuelType"].apply(
        lambda x: HELPER_fuel_correto(x, valid_list, threshold)
    )
    return df

In [ ]:
df=fuzzy_fuel(df)
df["fuelType"].unique()

In [ ]:
df[df["fuelType"].isna()]

In [ ]:
df["fuelType"].value_counts()

# **3. Numerical Variable Exploration and Cleaning**

## **3.1 Price distribution**

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["price"].dropna(), bins=50, edgecolor="black")
plt.title("Distribution of Car Prices")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df["price"].describe()

In [ ]:
ordenado = df.sort_values("price", ascending=True)

baratos = ordenado[["Brand", "model", "year", "price"]].head(5)
caros = baratos[["Brand", "model", "year", "price"]].tail(5)
display(baratos)
display(caros)
#acho possível estes carros em especifico terem estes preços

In [ ]:
def preco_no_fim(df):
    df = df.copy()

    df = df[[col for col in df.columns if col != "price"] + ["price"]]
    return df

## **3.2 Year analysis and cleaning**

In [ ]:
df["year"].unique()

In [ ]:
df[df["year"].isna()]

In [ ]:
old_cars = df[df["year"] < 2000][["Brand", "model", "year"]]
display(old_cars)
#os carros de 1970 nem existiam em 1970

In [ ]:
def limpar_anos(df):
    df = df.copy()

    df["year"] = np.round(df["year"]).astype("float")
    df.loc[(df["year"] < 1980) | (df["year"] > 2020),"year"] = np.nan
    df["year"] = df["year"].astype("Int64")
    return df

In [ ]:
df = limpar_anos(df) # call the function "limpar_anos(df)"
df[df["year"].isna()]

In [ ]:
df["year"].unique()

In [ ]:
sorted(df["year"].dropna().unique())


## **3.3 Mileage analysis and cleaning**

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["mileage"].dropna(), bins=50, edgecolor="black")
plt.title("Distribution of Mileage")
plt.xlabel("Mileage")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df[df["mileage"].isna()]

In [ ]:
df[df["mileage"] < 0]

In [ ]:
def impossible_to_nan(df, col, val=0, lower_upper="lower"):
    # Make a copy of the original DataFrame so the original data isn't modified directly
    df = df.copy()

    # Check whether we want to filter out values lower than 'val' or greater than 'val'
    if lower_upper == "lower":
        # If 'lower', replace all values in 'col' that are smaller than 'val' with NaN
        df.loc[df[col] < val, col] = np.nan
        # Return the cleaned DataFrame
        return df
    else:
        # If 'upper', replace all values in 'col' that are greater than 'val' with NaN
        df.loc[df[col] > val, col] = np.nan
        # Return the cleaned DataFrame
        return df


In [ ]:
df = impossible_to_nan(df, "mileage")

if USE_TOY:
    # Em toy dataset a última linha é 999
    print("\nTOY MODE → mostrando última linha para inspecção:")
    display(df.loc[df.index.max()])
else:
    print("\nFULL MODE → mostrando linha 70615 para inspecção:")
    display(df.loc[70615])


In [ ]:
df[df["mileage"] < 0]

In [ ]:
df[df["mileage"].isna()]

## **3.4 Tax analysis**

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["tax"].dropna(), bins=20, edgecolor="black")
plt.title("Distribution of tax")
plt.xlabel("tax")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df[df["tax"].isna()]

In [ ]:
def impossible_to_nan(df, col, val=0, lower_upper="lower"):
    df=df.copy()

    if lower_upper=="lower":
        df.loc[df[col]<val, col] = np.nan
        return df
    else:
        df.loc[df[col]>val, col] = np.nan
        return df

In [ ]:
df=impossible_to_nan(df, "tax")

plt.figure(figsize=(8, 5))
plt.hist(df["tax"].dropna(), bins=20, edgecolor="black")
plt.title("Distribution of tax")
plt.xlabel("tax")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df[df["tax"].isna()]

In [ ]:
g = df[df["tax"] == 0]
g["year"].value_counts()

## **3.5 mpg analysis**

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["mpg"].dropna(), bins=50, edgecolor="black")
plt.title("Distribution of mpg")
plt.xlabel("mpg")
plt.ylabel("Frequency")
plt.show()
#FIXE

In [ ]:
df[df["mpg"].isna()]

In [ ]:
df=impossible_to_nan(df, "mpg")

plt.figure(figsize=(8, 5))
plt.hist(df["mpg"].dropna(), bins=50, edgecolor="black")
plt.title("Distribution of mpg")
plt.xlabel("mpg")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df[df["mpg"].isna()]

## **3.5 Engine size analysis**

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["engineSize"].dropna(), bins=50, edgecolor="black")
plt.title("Distribution of engineSize")
plt.xlabel("engineSize")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df[df["engineSize"] < 0.6]

In [ ]:
df[df["engineSize"].isna()]

In [ ]:
df=impossible_to_nan(df, "engineSize", 0.49)
df[df["engineSize"].isna()]

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["engineSize"].dropna(), bins=50, edgecolor="black")
plt.title("Distribution of engineSize")
plt.xlabel("engineSize")
plt.ylabel("Frequency")
plt.show()

## **3.6 paintQuality analysis**

In [ ]:
df["paintQuality%"].describe()

In [ ]:
df[df["paintQuality%"] > 100]

In [ ]:
df[df["paintQuality%"].isna()]

In [ ]:
df = impossible_to_nan(df,"paintQuality%", 100, "upper")
df[df["paintQuality%"].isna()]

In [ ]:
df["paintQuality%"].describe()

## **3.7 previous Owners analysis**

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["previousOwners"].dropna(), bins=50, edgecolor="black")
plt.title("Distribution of previousOwners")
plt.xlabel("previousOwners")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df["previousOwners"].unique()

In [ ]:
df[df["previousOwners"]<0]

In [ ]:
df[df["previousOwners"].isna()]

In [ ]:
def round_owners_int(df):
    df=df.copy()
    df["previousOwners"] = pd.to_numeric(df["previousOwners"], errors="coerce")

    df['previousOwners'] = df['previousOwners'].round().astype('Int64')
    return df

In [ ]:
df=impossible_to_nan(df,"previousOwners")
df=round_owners_int(df)
df["previousOwners"].unique()

## **3.8 hasDamage analysis**

In [ ]:
df["hasDamage"].unique() #está cheia de zeros

In [ ]:
def remove_hasdmg(df):
    df = df.copy()

    df = df.drop(columns=['hasDamage'])
    return df

In [ ]:
df=remove_hasdmg(df)

In [ ]:
display(df)

# **4. MIssing Values exploration**

## **4.1 Missing Values Overview (EDA)**


In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percentage': (missing / len(df)) * 100
})

print("Missing Values with Percentage:\n")
print(missing_df)



## **4.2 Missing Data Handling Strategy**

In [ ]:
"""
Missing Data Handling Strategy

- Categorical variables: fill missing values with 'Unknown'
- Numerical variables: invalid or impossible values are converted to NaN
  (handled later during imputation in the ML pipeline)
- Cleaning functions are applied consistently across all datasets
"""


## **4.3 Helper Functions for Missing Values**

In [ ]:
def fill_cats_UNKNOWN(df, cats):
    df = df.copy()
    for column in cats:
        df[column] = df[column].fillna('Unknown')

    return df

## **4.4 Feature Grouping**

In [ ]:
num_cols = ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners']
cat_cols = ['Brand', 'model', 'transmission', 'fuelType']
int_cols = ['year', 'previousOwners']
float_cols = ['mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%']

## **4.5 Full Data Cleaning Pipeline**

In [ ]:
def clean_df(df, valid_models):
    df=df.copy()

    df=carID_como_index(df)
    df=fuzzy_marcas(df)
    df=fuzzy_modelos(df, valid_models)
    df=inferir_marca_com_modelo(df, valid_models)
    df=limpar_anos(df)
    df=fuzzy_transmissao(df)
    df=impossible_to_nan(df, "mileage")
    df=fuzzy_fuel(df)
    df=impossible_to_nan(df, "tax")
    df=impossible_to_nan(df, "mpg")
    df=impossible_to_nan(df, "engineSize", 0.49)
    df=impossible_to_nan(df,"paintQuality%", 100, "upper")
    df=impossible_to_nan(df,"previousOwners")
    df=round_owners_int(df)
    df=remove_hasdmg(df)
    df=fill_cats_UNKNOWN(df,cat_cols)

    return df

## **4.6 Data Cleaning Execution & Validation**


In [ ]:
df=fill_cats_UNKNOWN(df, cat_cols)

In [ ]:
og_df=clean_df(og_df, valid_models)

In [ ]:
assert pd.testing.assert_frame_equal(df, og_df) is None

# **5. DATA SPLIT & PREPROCESSING PIPELINE**



### Pipeline Strategy

This preprocessing pipeline was designed to strictly prevent data leakage and
to mirror a real-world deployment scenario. All statistics required for data
transformation (imputation values, outlier thresholds, scaling parameters) are
learned exclusively from the training set and then reused unchanged on the
validation data.

The pipeline follows a clear and reproducible order:
1. Feature–target separation
2. Train–validation split
3. Training-based preprocessing (imputation, outlier handling, scaling)
4. Consistent application of learned transformations to unseen data

This approach ensures fair model evaluation and reliable generalisation
performance.

All exploratory plots are generated exclusively from the training set to avoid
any influence from unseen validation data.



## **5.1 Feature–Target Separation**

In [ ]:
def separar_y(df):
    df=df.copy()

    X = df.drop('price', axis = 1)
    y = df['price']

    return X, y

In [ ]:
X, y = separar_y(df)
X

In [ ]:
y

## **5.2 Train–Validation Split**

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X,y, test_size = 0.3, random_state = RANDOM_SEED, shuffle = True)

## **5.3 Missing Value Imputation (Numerical Features)**

### **5.3.1 Imputation Strategy Definition**

In [ ]:
def fill_nans(X, ints, floats, fill_values=None):
    X = X.copy()

    if fill_values is None:
        fill_values = {"float": {}, "int": {}}
        for column in floats:
            mean_to_fill = X[column].mean()
            X[column] = X[column].fillna(mean_to_fill)
            fill_values["float"][column] = mean_to_fill

        for column in ints:
            median_to_fill = X[column].median()
            X[column] = X[column].fillna(median_to_fill).astype("Int64")
            fill_values["int"][column] = median_to_fill

        return X, fill_values
    else:
        for col in floats:
            X[col] = X[col].fillna(fill_values["float"][col])
        for col in ints:
            X[col] = X[col].fillna(fill_values["int"][col]).astype("Int64")

        return X

### **5.3.2 Application to Train and Validation Sets**

In [ ]:
X_train, fill_values = fill_nans(X_train, int_cols, float_cols)
X_val = fill_nans(X_val, int_cols, float_cols, fill_values)

In [ ]:
X_train.info()

In [ ]:
X_val.info()

## **5.4 Exploratory Analysis of Numerical Features**

In [ ]:
def plot_nums(X, num_cols):
    for col in num_cols:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        fig.suptitle(col, fontsize=14, fontweight='bold')

        # Boxplot
        sns.boxplot(y=X[col], ax=axes[0], color='skyblue')
        axes[0].set_title("Boxplot")

        # Histogram
        sns.histplot(X[col], kde=True, ax=axes[1], color='salmon')
        axes[1].set_title("Histogram")

        plt.tight_layout()
        plt.show()

In [ ]:
plot_nums(X_train, num_cols)

## **5.5 Outlier Treatment & Skewness Correction**


### **5.5.1 Outlier & Skewness Strategy (Training Phase)**

In [ ]:
def outliers_skews_train(X):
    """
    Purpose:
        Apply outlier correction and skew reduction transformations to the
        training dataset, while recording all thresholds and operations used
        so that the exact same transformations can be applied later to the
        validation and test datasets without recalculating parameters.

    Overview of Operations:
        - mileage:
            Apply log1p transformation to reduce skewness in long-tailed
            distributions common in car mileage data.
        - tax:
            Identify extreme values using the 97.5th percentile and cap all
            values above this threshold (upper clipping).
        - mpg:
            Apply similar upper clipping based on the 97.5th percentile to
            remove unrealistically high fuel efficiency values.
        - engineSize:
            Apply bilateral clipping using the 1st and 99th percentiles to
            remove extreme small/large engine sizes while preserving most data.

    Returns:
        X (pd.DataFrame):
            A transformed copy of the dataset where outliers and skew-heavy
            features have been adjusted.
        outlier_info (dict):
            A dictionary containing all thresholds and transformation flags
            required to reproduce the same preprocessing steps during inference.

    Notes:
        This function must be used only on the training data. The resulting
        outlier_info dictionary should be passed to the corresponding
        outliers_skews_test() function to ensure consistent preprocessing of
        new or unseen data, avoiding data leakage.
    """

    X = X.copy()
    outlier_info = {}

    if "mileage" in X.columns:
        X["mileage"] = np.log1p(X["mileage"])
        outlier_info["mileage"] = {"log_transform": True}

    if "tax" in X.columns:
        upper = X["tax"].quantile(0.975)
        X["tax"] = X["tax"].clip(upper=upper)
        outlier_info["tax"] = {"upper": upper}

    if "mpg" in X.columns:
        upper = X["mpg"].quantile(0.975)
        X["mpg"] = X["mpg"].clip(upper=upper)
        outlier_info["mpg"] = {"upper": upper}

    if "engineSize" in X.columns:
        lower = X["engineSize"].quantile(0.01)
        upper = X["engineSize"].quantile(0.99)
        X["engineSize"] = X["engineSize"].clip(lower=lower, upper=upper)
        outlier_info["engineSize"] = {"lower": lower, "upper": upper}

    return X, outlier_info

### **5.5.2 Consistent Application to Validation/Test Data**

In [ ]:
def outliers_skews_test(X, outlier_info):
    """
    Purpose:
        Apply the exact same outlier and skew corrections used during training
        to a new dataset (validation or test) using the thresholds and
        transformation rules previously recorded in outlier_info.
        No new statistics are computed here to avoid data leakage.

    Overview of Operations:
        - mileage:
            If log1p transformation was applied during training, apply the same
            transformation to preserve consistency in feature scaling.
        - tax:
            Apply upper clipping using the threshold recorded in outlier_info.
        - mpg:
            Apply upper clipping using the training-derived threshold.
        - engineSize:
            Apply bilateral clipping using the stored lower and upper bounds.

    Inputs:
        X (pd.DataFrame):
            New data (validation or test) to preprocess.
        outlier_info (dict):
            Dictionary containing all thresholds and transformation flags
            computed during training by outliers_skews_train().

    Returns:
        X (pd.DataFrame):
            A transformed copy of the dataset with the same preprocessing rules
            applied as during training, guaranteeing consistent feature scaling
            and preventing data leakage.

    Notes:
        This function must not compute new thresholds (quantiles, means, etc.).
        Its only responsibility is to enforce the transformations defined
        during training, ensuring reproducibility and consistent model behavior.
    """

    X = X.copy()

    if "mileage" in outlier_info and "mileage" in X.columns:
        if outlier_info["mileage"].get("log_transform", False):
            X["mileage"] = np.log1p(X["mileage"])

    if "tax" in outlier_info and "tax" in X.columns:
        X["tax"] = X["tax"].clip(upper=outlier_info["tax"]["upper"])

    if "mpg" in outlier_info and "mpg" in X.columns:
        X["mpg"] = X["mpg"].clip(upper=outlier_info["mpg"]["upper"])

    if "engineSize" in outlier_info and "engineSize" in X.columns:
        X["engineSize"] = X["engineSize"].clip(
            lower=outlier_info["engineSize"]["lower"],
            upper=outlier_info["engineSize"]["upper"]
        )

    return X


### **5.5.3 Application of Outlier & Skewness Corrections**

In [ ]:
X_train, outlier_info = outliers_skews_train(X_train)
X_val = outliers_skews_test(X_val, outlier_info)

In [ ]:
X_train_num = X_train[num_cols].copy()
X_train_cat = X_train[cat_cols].copy()

X_val_num = X_val[num_cols].copy()
X_val_cat = X_val[cat_cols].copy()

## **5.6 Feature Scaling (Numerical Variables)**

In [ ]:
scaler = MinMaxScaler()
scaler.fit(X_train_num)
X_train_num_scaled = scaler.transform(X_train_num)
X_train_num_scaled

In [ ]:
X_train_num_scaled = pd.DataFrame(X_train_num_scaled, columns = X_train_num.columns).set_index(X_train.index)
X_train_num_scaled

In [ ]:
X_val_num_scaled = scaler.transform(X_val_num)
X_val_num_scaled = pd.DataFrame(X_val_num_scaled, columns = X_val_num.columns).set_index(X_val.index)
X_val_num_scaled

## **5.7 Preprocessing Summary**

At the end of this preprocessing stage, the dataset is fully prepared for model
training. Missing values were handled using training-derived statistics, highly
skewed variables were stabilised, and extreme outliers were clipped based on
robust quantile thresholds.

All numerical features were subsequently normalised using Min–Max scaling to
ensure comparable feature ranges and to support models sensitive to scale.

Most importantly, every transformation applied to the validation data strictly
reused parameters learned from the training set, guaranteeing a leakage-free
pipeline and realistic performance estimates.

The resulting processed datasets are now suitable for direct integration into
the modelling pipeline.


# **6 FEATURE SELECTION**

## **6.1 Correlation**

In [ ]:
corr = X_train_num_scaled.assign(price=y_train).corr(method='spearman')
def cor_heatmap(cor):
    plt.figure(figsize=(12,10))
    sns.heatmap(data = cor, annot = True, cmap = plt.cm.Reds, fmt='.1')
    plt.show()
cor_heatmap(corr)

## **6.2 Estimating Numerical Feature Relevance Using Mutual Information Scores**

In [ ]:
mi_scores = mutual_info_regression(X_train_num_scaled, y_train, random_state=0)
mi_scores = pd.Series(mi_scores, index=X_train_num_scaled.columns)
mi_scores.sort_values(ascending=False, inplace=True)
print(mi_scores)

## **6.3 Chi-Square Test for Categorical Predictor Significance**

In [ ]:
def TestIndependence(X,y,var,alpha=0.05):
    dfObserved = pd.crosstab(y,X)
    chi2, p, dof, expected = stats.chi2_contingency(dfObserved.values)
    dfExpected = pd.DataFrame(expected, columns=dfObserved.columns, index = dfObserved.index)
    if p<alpha:
        result="{0} is IMPORTANT for Prediction".format(var)
    else:
        result="{0} is NOT an important predictor. (Discard {0} from model)".format(var)
    print(result)

for var in X_train_cat:
    TestIndependence(X_train_cat[var],y_train, var)

## **6.4 Cramér’s V: Association Metric for Categorical Predictors**

In [ ]:
def cramers_v(x, y, var):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    result =  np.sqrt(chi2 / (n * (min(k - 1, r - 1))))
    print(f"{var}: {result}")

for var in X_train_cat:
    cramers_v(X_train_cat[var],y_train, var)

## **6.5 Mutual Information for Categorical Variables via Label Encoding**

In [ ]:
def mutual_info(X, y, var):
    le = LabelEncoder()
    X_encoded = le.fit_transform(X.astype(str)).reshape(-1, 1)

    mi = mutual_info_classif(X_encoded, y, discrete_features=True, random_state=0)
    print(f"{var}: {mi[0]:.4f}")

for var in X_train_cat:
    mutual_info(X_train_cat[var],y_train, var)

## **6.6 Teste ANOVA para Avaliar a Influência das Variáveis Categóricas na Variável Alvo**

In [ ]:
def anova(X_col, y, var):
    groups = [y[X_col == level] for level in X_col.dropna().unique()]

    # Perform one-way ANOVA
    f_stat, p_val = f_oneway(*groups)

    print(f"{var.upper()}")
    print(f"  F-stat:  {f_stat:.3f}")
    print(f"  p-value: {p_val:.6f}")
    print("\n")  # line spacing between variables

    return pd.Series({"F-stat": f_stat, "p-value": p_val}, name=var)


for var in X_train_cat:
    anova(X_train_cat[var],y_train, var)

In [ ]:
"""ANOVA says: “These features cause significant average price differences.”

MI says: “These features explain uncertainty in price (model gives the most info).”

Cramer’s V says: “These variables have moderate categorical association strength.”"""

## **6.7 Seleção de Variáveis Numéricas com RFE e Avaliação do RMSE**

In [ ]:
model = LinearRegression()

In [ ]:
rfe = RFE(estimator = model, n_features_to_select = 4)

In [ ]:
X_rfe = rfe.fit_transform(X = X_train_num_scaled, y = y_train)

In [ ]:
X_train_num_scaled.columns

In [ ]:
rfe.support_

In [ ]:
rfe.ranking_

In [ ]:
selected_features = pd.Series(rfe.support_, index = X_train_num_scaled.columns)
selected_features

In [ ]:
print(X_train_num_scaled.shape)
print(X_val_num_scaled.shape)
print(y_train.shape)
print(y_val.shape)


In [ ]:
nof_list = np.arange(1, len(X_train_num_scaled.columns) + 1)
best_score = np.inf
best_n = 0

train_score_list = []
val_score_list = []

for n in nof_list:
    model = LinearRegression()
    rfe = RFE(estimator=model, n_features_to_select=n)

    # Fit RFE on training data
    X_train_rfe = rfe.fit_transform(X_train_num_scaled, y_train)
    X_val_rfe = rfe.transform(X_val_num_scaled)

    # Fit model
    model.fit(X_train_rfe, y_train)

    # Predict
    y_train_pred = model.predict(X_train_rfe)
    y_val_pred = model.predict(X_val_rfe)

    # Compute RMSE manually
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

    # Store negated RMSE (so higher = better for plotting)
    train_score_list.append(-train_rmse)
    val_score_list.append(-val_rmse)

    # Track best score (lowest RMSE)
    if val_rmse < best_score:
        best_score = val_rmse
        best_n = n

print(f"Optimum number of features: {best_n}")
print(f"Lowest validation RMSE ({best_n} features): {best_score:.4f}")


In [ ]:
plt.plot(nof_list, [-s for s in train_score_list], label='Train RMSE', color='lightgreen')
plt.plot(nof_list, [-s for s in val_score_list], label='Validation RMSE', color='gray')
plt.xlabel('Number of features')
plt.ylabel('RMSE')
plt.legend()
plt.show()

"""Acho que há colinearidade entre variáveis como vimos antes, acho 4 sao suifciente"""

## **6.8 Seleção de Variáveis com LassoCV**

In [ ]:
reg = LassoCV()

In [ ]:
reg.fit(X_train_num_scaled, y_train)

In [ ]:
coef = pd.Series(reg.coef_, index = X_train_num_scaled.columns)
coef.sort_values()

In [ ]:
print("Lasso picked " + str(sum(coef != 0)) + " variables and eliminated the other " +  str(sum(coef == 0)) + " variables")

In [ ]:
def plot_importance(coef,name):
    imp_coef = coef.sort_values()
    plt.figure(figsize=(8,10))
    imp_coef.plot(kind = "barh")
    plt.title("Feature importance using " + name + " Model")
    plt.show()

In [ ]:
# CODE HERE
plot_importance(coef,'Lasso')

## **6.9 Feature Selection Summary: Numerical and Categorical Predictors**


Numerical Data
| Feature            | Spearman | Mutual Info | RFE (LR) | Lasso | Decision     |
| ------------------ | -------- | ----------- | -------- | ----- | ------------ |
| **year**           | ✅        | ✅           | ✅        | ✅     | **Keep**     |
| **mileage**        | ❌        | ✅           | ✅        | ❌     | **Keep**     |
| **tax**            | ✅        | ❌           | ❌        | ❌     | Try          |
| **mpg**            | ❌        | ✅           | ✅        | ✅     | **Keep**     |
| **engineSize**     | ✅        | ✅           | ✅        | ✅     | **Keep**     |
| **paintQuality%**  | ❌        | ❌           | ❌        | ❌     | Discard      |
| **previousOwners** | ❌        | ❌           | ❌        | ❌     | Discard      |

Categorical Data
| Feature          | Chi-Square | Cramér’s V | ANOVA F-test | Decision |
| ---------------- | ---------- | ---------- | ------------ | -------- |
| **brand**        | ✅          | ✅          | ✅            | **Keep** |
| **model**        | ✅          | ✅          | ✅            | **Keep** |
| **transmission** | ✅          | ✅          | ✅            | **Keep** |
| **fuelType**     | ❌          | ❌          | ✅            | Discard  |

Conclusion:

The most predictive features for our car price model appear to be:

Numerical: year, engineSize, mpg

Categorical: brand, model, transmission


# **7. MODEL SELECTION**

### Model Selection Strategy

This section implements a controlled and reproducible model selection
procedure. A fixed train–validation split is enforced via PredefinedSplit to
ensure that all candidate models are evaluated on the exact same validation
set, mirroring a Kaggle-style evaluation setup.

A unified, leakage-safe preprocessing function is used throughout the model
selection process, guaranteeing that all transformations are learned only
from training data and consistently applied during validation.

Different preprocessing rules (notably outlier handling) are conditionally
applied depending on the model family, reflecting known inductive biases of
tree-based versus linear and neural models.


## **7.0 Initial Setup**


In [ ]:
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor


RANDOM_SEED = 1907



## **7.1 Train–Validation Split and Validation Strategy**

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.3,
    shuffle=True,
    random_state=RANDOM_SEED
)

# PredefinedSplit para forçar o mesmo validation set durante RandomizedSearchCV
test_fold = np.concatenate([
    -1 * np.ones(len(X_train)),
     0 * np.ones(len(X_val))
])

ps = PredefinedSplit(test_fold)

# --------------------------------------------------
# Target vector aligned with PredefinedSplit
# --------------------------------------------------
y_train_val_full = pd.concat(
    [y_train, y_val],
    axis=0
).reset_index(drop=True)



## **7.2 Unified Custom Preprocessing (Leakage-Safe)**

In [ ]:
def preprocess_for_search(
    X,
    int_cols,
    float_cols,
    cat_cols,
    ohe,
    apply_outliers=True,
    fit=True,
    fill_values=None,
    outlier_info=None
):
    """
    Unified, leakage-safe preprocessing pipeline used for:
    - model selection
    - final training
    - validation / test prediction

    Steps:
    1. Numerical missing value imputation (leakage-safe)
    2. Optional outlier & skew correction
    3. One-hot encoding (categorical variables)

    Returns:
    - X_final: preprocessed feature matrix
    - artifacts: dict with learned preprocessing objects
    """

    X = X.copy()
    artifacts = {}

    # --------------------------------------------------
    # 1. Numerical imputation (USING EXISTING fill_nans)
    # --------------------------------------------------
    if fit:
        X, fill_values = fill_nans(X, int_cols, float_cols)
        artifacts["fill_values"] = fill_values
    else:
        X = fill_nans(X, int_cols, float_cols, fill_values)

    # --------------------------------------------------
    # 2. Optional outlier & skew correction
    # --------------------------------------------------
    if apply_outliers:
        if fit:
            X, outlier_info = outliers_skews_train(X)
            artifacts["outlier_info"] = outlier_info
        else:
            X = outliers_skews_test(X, outlier_info)

    # --------------------------------------------------
    # 3. One-Hot Encoding (categorical)
    # --------------------------------------------------
    if fit:
        ohe.fit(X[cat_cols])

    encoded = ohe.transform(X[cat_cols])
    encoded_cols = ohe.get_feature_names_out(cat_cols)

    X_encoded = pd.DataFrame(
        encoded,
        columns=encoded_cols,
        index=X.index
    )

    # --------------------------------------------------
    # 4. Rebuild final dataset
    # --------------------------------------------------
    X_final = pd.concat(
        [X.drop(columns=cat_cols), X_encoded],
        axis=1
    )

    X_final.columns = X_final.columns.astype(str)

    return X_final, artifacts



## **7.3 Candidate Models and Hyperparameter Search Spaces**

In [ ]:
## 7.3 — Modelos e Espaços de Hiperparâmetros

models = {
    "RandomForest": RandomForestRegressor(
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),

    "GradientBoosting": GradientBoostingRegressor(
        random_state=RANDOM_SEED
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        random_state=RANDOM_SEED
    ),

    "ExtraTrees": ExtraTreesRegressor(
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),

    "MLP": MLPRegressor(
        random_state=RANDOM_SEED,
        max_iter=400,
        early_stopping=True,
        n_iter_no_change=10,
        validation_fraction=0.1,
        solver="adam",
        learning_rate_init=0.001
    ),

    "Ridge": Ridge()
}


param_spaces = {
    "RandomForest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt", "log2"]
    },

    "GradientBoosting": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [2, 3, 4]
    },

    "HistGradientBoosting": {
        "max_iter": [100, 200, 300],
        "learning_rate": [0.05, 0.1],
        "max_depth": [None, 10, 20],
        "min_samples_leaf": [20, 50]
    },

    "ExtraTrees": {
        "n_estimators": [200, 400, 600],
        "max_depth": [None, 20, 30],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt", "log2"]
    },

    "MLP": {
        "hidden_layer_sizes": [(32,), (64,), (32, 32)],
        "alpha": [0.0001, 0.001, 0.01]
    },

    "Ridge": {
        "alpha": [0.1, 1.0, 10.0]
    }
}



## **7.4 RandomizedSearchCV with Controlled Validation**


In [ ]:
results = []

print("\n===== MODEL SELECTION STARTED =====\n")

for name, model in models.items():
    print(f"\n### Running: {name} ###")

    # --------------------------------------------------
    # 7.5.1 — Outlier handling rule by model type
    # --------------------------------------------------
    if name in ["RandomForest", "ExtraTrees", "GradientBoosting", "HistGradientBoosting"]:
        apply_outliers = False   # tree-based models
    else:
        apply_outliers = True    # Ridge / MLP

    # --------------------------------------------------
    # 7.5.2 — Leakage-safe preprocessing
    #   fit  -> TRAIN only
    #   apply -> VALIDATION
    # --------------------------------------------------
    ohe_model = OneHotEncoder(
        drop="first",
        sparse_output=False,
        handle_unknown="ignore"
    )

    # ---- FIT preprocessing on TRAIN ----
    X_train_proc, artifacts = preprocess_for_search(
        X_train,
        int_cols=int_cols,
        float_cols=float_cols,
        cat_cols=cat_cols,
        ohe=ohe_model,
        apply_outliers=apply_outliers,
        fit=True
    )

    # --------------------------------------------------
    # Sanity checks on learned artifacts
    # --------------------------------------------------
    assert "fill_values" in artifacts, "Missing fill_values in preprocessing artifacts"
    if apply_outliers:
        assert "outlier_info" in artifacts, "Missing outlier_info while apply_outliers=True"

    # ---- APPLY preprocessing to VALIDATION ----
    X_val_proc, _ = preprocess_for_search(
        X_val,
        int_cols=int_cols,
        float_cols=float_cols,
        cat_cols=cat_cols,
        ohe=ohe_model,
        apply_outliers=apply_outliers,
        fit=False,
        fill_values=artifacts["fill_values"],
        outlier_info=artifacts.get("outlier_info")
    )

    # ---- Rebuild dataset for PredefinedSplit ----
    X_preprocessed_model = pd.concat(
        [X_train_proc, X_val_proc],
        axis=0
    ).reset_index(drop=True)

    # --------------------------------------------------
    # 7.5.3 — RandomizedSearchCV
    # --------------------------------------------------
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_spaces[name],
        n_iter=15,
        cv=ps,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
        verbose=1,
        random_state=RANDOM_SEED
    )

    search.fit(X_preprocessed_model, y_train_val_full.values.ravel())

    best_mae = -search.best_score_

    print(f"{name} — Validation MAE: {best_mae:.4f}")
    print("Best params:", search.best_params_)
    print("Outlier handling:", apply_outliers)

    # --------------------------------------------------
    # 7.5.4 — Store results
    # --------------------------------------------------
    results.append({
        "model": name,
        "val_mae": best_mae,
        "best_params": search.best_params_,
        "estimator": search.best_estimator_,
        "apply_outliers": apply_outliers
    })

print("\n===== MODEL SELECTION COMPLETE =====")

results_sorted = sorted(results, key=lambda x: x["val_mae"])
best = results_sorted[0]

print("\n### BEST MODEL ###")
print(best)








### Step 1 — Save the Model Selection Artifacts

After completing the model selection phase, the following critical artifacts are defined and must be preserved (either mentally or documented in markdown), as they will be used consistently in the subsequent stages of the pipeline:

- Winning model  
```python
best["model"] == "ExtraTrees"
```

- Best hyperparameters found


```python
best["best_params"]
```
- Associated preprocessing rule


```python
apply_outliers = False
```

# **8. EXTRA TREES — FINAL HYPERPARAMETER TUNING**

## **8.0 ExtraTrees Final Tuning — Strategy**

ExtraTrees Final Tuning Strategy

This section performs a focused, final hyperparameter optimization of the
ExtraTrees model selected during model comparison. A reduced but expressive
search space is used to balance performance gains and computational cost.

All preprocessing steps remain strictly leakage-safe, with transformations
learned exclusively from training data and reused consistently for validation
and test inference. The final model is retrained on the full dataset and
stabilized via an increased number of trees to reduce variance prior to
submission.


## **8.1 Extra Trees Final Tunning - Initial Setup**

In [ ]:
# ==========================================================
# 8.0 — EXTRA TREES FINAL TUNING: SETUP
# ==========================================================

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
import numpy as np

RANDOM_SEED = 1907

# Garantir que usamos X_train, X_val, y_train, y_val criados antes
print("Training size:", len(X_train))
print("Validation size:", len(X_val))

# PredefinedSplit para garantir validação consistente
test_fold = np.concatenate([
    -1 * np.ones(len(X_train)),   # treino
     0 * np.ones(len(X_val))      # validação
])

ps = PredefinedSplit(test_fold)

print("PredefinedSplit ready.")


## **8.2 Preprocessing for ExtraTrees**

In [ ]:
# ==========================================================
# 8.3 — LEAKAGE-SAFE PREPROCESSING FOR EXTRATREES (TRAIN → VAL)
# ==========================================================

# Dedicated OneHotEncoder (fit ONLY on TRAIN)
ohe_extra = OneHotEncoder(
    drop="first",
    sparse_output=False,
    handle_unknown="ignore"
)

# --------------------------------------------------
# 1) FIT preprocessing ONLY on TRAIN
# --------------------------------------------------
X_train_proc_extra, artifacts_extra = preprocess_for_search(
    X_train,
    int_cols=int_cols,
    float_cols=float_cols,
    cat_cols=cat_cols,
    ohe=ohe_extra,
    apply_outliers=False,   # ExtraTrees → no outlier handling
    fit=True
)

# --------------------------------------------------
# 2) APPLY preprocessing to VALIDATION (NO FIT)
# --------------------------------------------------
X_val_proc_extra, _ = preprocess_for_search(
    X_val,
    int_cols=int_cols,
    float_cols=float_cols,
    cat_cols=cat_cols,
    ohe=ohe_extra,  # reuse fitted encoder
    apply_outliers=False,
    fit=False,
    fill_values=artifacts_extra["fill_values"],
    outlier_info=artifacts_extra.get("outlier_info")
)

# --------------------------------------------------
# 3) Rebuild dataset for PredefinedSplit
# --------------------------------------------------
X_preproc_extra = pd.concat(
    [X_train_proc_extra, X_val_proc_extra],
    axis=0
).reset_index(drop=True)

y_train_val_full = pd.concat(
    [y_train, y_val],
    axis=0
).reset_index(drop=True)

print("Preprocessing completed (leakage-safe).")
print("Shape:", X_preproc_extra.shape)
print("Artifacts available:", list(artifacts_extra.keys()))




## **8.3 Final Hyperparameter Search (RandomizedSearchCV)**

In [ ]:
# ================================================================
# 8.3 — RANDOMIZEDSEARCHCV FINAL PARA EXTRATREES
# ================================================================
"""
README
------
Esta secção executa a pesquisa final de hiperparâmetros para o modelo ExtraTrees,
usando um espaço de procura equilibrado (Opção C).
Objetivos:
 - Reduzir o tempo total de treino (≈20–25 min)
 - Manter desempenho semelhante ao RandomizedSearchCV completo
 - Evitar configurações demasiado pesadas (ex.: max_features='log2', max_depth demasiado ampla)

A validação continua a ser feita com o PredefinedSplit definido anteriormente.
"""

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import RandomizedSearchCV

# ================================================================
# 8.3.1 Base Model Definition
# ================================================================
extra_final_model = ExtraTreesRegressor(
    random_state=RANDOM_SEED,
    n_jobs=-1
)

# ================================================================
# 8.3.2 Optimized Hyperparameter Space
# ================================================================
extra_param_space_C = {
    "n_estimators": [200, 400, 600],     # quantidade de árvores
    "max_depth": [None, 30],             # profundidade controlada
    "min_samples_split": [2, 5],         # divisão mínima
    "min_samples_leaf": [1, 2],          # folhas mínimas
    "max_features": ["sqrt"]             # mais rápido e estável
}

# ================================================================
# 8.3.3 RandomizedSearchCV Configuration
# ================================================================
print("\n==== EXTRA TREES — EXPANDED HYPERPARAMETER SEARCH (OPTION C) ====\n")
print("Running RandomizedSearchCV with optimized search space...\n")

search_extra_C = RandomizedSearchCV(
    estimator=extra_final_model,
    param_distributions=extra_param_space_C,
    n_iter=30,                                 # → reduz tempo pela metade
    cv=ps,                                     # PredefinedSplit
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=2,
    random_state=RANDOM_SEED
)

# ================================================================
# 8.3.4 Search Execution and Results
# ================================================================
search_extra_C.fit(X_preproc_extra, y_train_val_full)

best_mae_C = -search_extra_C.best_score_
best_params_C = search_extra_C.best_params_

# ================================================================
# 8.3.5 — Printing results
# ================================================================
print("\n===== BEST RESULT — EXTRA TREES (OPTION C) =====")
print("Best MAE:", best_mae_C)
print("Best Params:", best_params_C)



## **8.4 Retraining ExtraTrees on the Full Dataset**

In [ ]:
# ==========================================================
# 8.4 — RE-TREINO FINAL DO EXTRA TREES (VARIANCE REDUCTION)
# ==========================================================
"""
README
------
Nesta célula, o modelo ExtraTrees vencedor é re-treinado
no DATASET COMPLETO (X, y), mantendo todos os hiperparâmetros
estruturais encontrados na fase de model selection.

A única alteração intencional é o aumento de n_estimators,
com o objetivo de reduzir variância e estabilizar previsões,
sem alterar a decisão do modelo vencedor.

Esta etapa NÃO faz parte da model selection.
É uma otimização final antes da submissão.
"""

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.preprocessing import OneHotEncoder

# --------------------------------------------------
# Parâmetros vencedores (da model selection)
# --------------------------------------------------
best_params_extra = {
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt"
}

# --------------------------------------------------
# Encoder FINAL (fit apenas aqui)
# --------------------------------------------------
final_ohe = OneHotEncoder(
    drop="first",
    sparse_output=False,
    handle_unknown="ignore"
)

print("Preprocessing full training data...")

# --------------------------------------------------
# Pré-processar TODO o dataset de treino (fit=True)
# --------------------------------------------------
X_full_processed_extra, artifacts_extra = preprocess_for_search(
    X,
    int_cols=int_cols,
    float_cols=float_cols,
    cat_cols=cat_cols,
    ohe=final_ohe,
    apply_outliers=False,   # árvores → sem outliers
    fit=True
)

# Guardar colunas usadas no treino (CRÍTICO para alinhar o test)
train_feature_cols_extra = X_full_processed_extra.columns.tolist()

# --------------------------------------------------
# Re-treino com mais árvores (variance reduction)
# --------------------------------------------------
final_extra = ExtraTreesRegressor(
    random_state=RANDOM_SEED,
    n_jobs=-1,
    n_estimators=1000,   # aumento deliberado
    **best_params_extra
)

print("Training final ExtraTrees model...")
final_extra.fit(X_full_processed_extra, y.values.ravel())

print("Final ExtraTrees model trained successfully.")
print("Number of trees:", final_extra.n_estimators)
print("Total training features:", len(train_feature_cols_extra))



## **8.5 Test Set Preprocessing (Inference Only)**

In [ ]:
# ==========================================================
# 8.5 — PREPARING TEST SET (NO FIT!)
# ==========================================================
"""
README
------
O test.csv é transformado usando EXATAMENTE os artifacts
aprendidos no treino final do ExtraTrees.
"""

from google.colab import files

# --------------------------------------------------
# Load test.csv
# --------------------------------------------------
uploaded = files.upload()
test_df = pd.read_csv(list(uploaded.keys())[0])

# --------------------------------------------------
# 1) Clean inicial
# --------------------------------------------------
test_clean = clean_df(test_df.copy(), valid_models)

# --------------------------------------------------
# 2) Preprocess SEM FIT (reuse artifacts)
# --------------------------------------------------
X_test_processed_extra, _ = preprocess_for_search(
    test_clean,
    int_cols=int_cols,
    float_cols=float_cols,
    cat_cols=cat_cols,
    ohe=ohe_extra,                             # encoder treinado
    apply_outliers=False,
    fit=False,
    fill_values=artifacts_extra["fill_values"],
    outlier_info=artifacts_extra.get("outlier_info")
)

# --------------------------------------------------
# 3) Alinhar features
# --------------------------------------------------
X_test_processed_extra = X_test_processed_extra.reindex(
    columns=train_feature_cols_extra,
    fill_value=0
)

print("Test processed successfully")
print("Test shape:", X_test_processed_extra.shape)


## **8.6 Final Test Predictions**

In [ ]:
# ==========================================================
# 8.6 — FINAL PREDICTIONS WITH EXTRA TREES
# ==========================================================
"""
README
------
Predictions on the processed test dataset using the ExtraTrees
model retrained on the full training data.

Guarantees:
- Column alignment
- No NaNs
- Deterministic inference
"""

# --------------------------------------------------
# Sanity checks
# --------------------------------------------------
assert "X_test_processed_extra" in globals(), "Test data not found"
assert "final_extra" in globals(), "Final model not found"

print("Test matrix shape:", X_test_processed_extra.shape)
print("Contains NaNs?", X_test_processed_extra.isna().sum().sum())

# --------------------------------------------------
# Predictions
# --------------------------------------------------
y_pred_extra = final_extra.predict(X_test_processed_extra)

print("Predictions completed successfully.")
print("Sample predictions:", y_pred_extra[:10])
print("Min prediction:", y_pred_extra.min())
print("Max prediction:", y_pred_extra.max())



## **8.7 — Exportar “submission_extra_trees.csv”**

In [ ]:
# ==========================================================
# 8.7 — EXPORTAR RESULTADOS EM CSV
# ==========================================================

submission_extra = pd.DataFrame({
    "carID": test_clean.index,
    "price": y_pred_extra
})

submission_extra.to_csv("submission_extra_trees.csv", index=False)

print("Saved submission_extra_trees.csv")
display(submission_extra.head())



# **9. MODEL BLENDING (EXTRATREES + RANDOM FOREST)**

## **9.0 Model Blending Strategy**

This section implements a simple yet effective model blending approach by
combining the predictions of two strong tree-based regressors: **ExtraTrees**
and **Random Forest**.

Both models are trained using the same leakage-safe preprocessing pipeline and
share an identical feature space, ensuring that their predictions are directly
comparable. While ExtraTrees typically offers lower bias, Random Forest tends to
provide more stable predictions. Blending their outputs helps reduce variance
and improve generalisation performance on unseen data.

Two blending strategies are evaluated:
1. **Simple averaging**, which provides a robust baseline ensemble.
2. **Weighted averaging**, which allows emphasising the stronger model while
   still benefiting from ensemble diversity.

All blending operations are performed strictly at inference time, without any
additional model fitting. The final blended predictions are then exported in a
Kaggle-compliant submission format.


In [ ]:
# ==========================================================
# Load test.csv and preserve IDs (GLOBAL, ONCE)
# ==========================================================

test_df = pd.read_csv("test.csv")

# Kaggle-safe immutable IDs
test_ids = test_df["carID"].values

# Clean test data
test_clean = clean_df(test_df.copy(), valid_models)

print("Test data loaded.")
print("Test shape:", test_clean.shape)


## **9.1 Test Set Preprocessing (Prediction Only)**

In [ ]:
"""
README
------
Prepare test set using:
- SAME preprocessing as training
- NO fitting
- SAME feature space
"""

# --------------------------------------------------
# Sanity checks
# --------------------------------------------------
assert "final_ohe" in globals(), "Final OneHotEncoder not found"
assert "train_feature_cols_extra" in globals(), "Training feature columns not found"
assert "final_extra" in globals(), "Final ExtraTrees model not found"
assert "artifacts_extra" in globals(), "ExtraTrees training artifacts not found"
assert "test_ids" in globals(), "test_ids not found"

print("Preprocessing test set for blending...")


# --------------------------------------------------
# Preprocess test set (NO FIT!)
# --------------------------------------------------
X_test_tmp_extra, _ = preprocess_for_search(
    test_clean,
    int_cols=int_cols,
    float_cols=float_cols,
    cat_cols=cat_cols,
    ohe=final_ohe,                        #  FINAL encoder (trained on full data)
    apply_outliers=False,                 # tree-based model
    fit=False,
    fill_values=artifacts_extra["fill_values"],
    outlier_info=artifacts_extra.get("outlier_info")
)

# --------------------------------------------------
# Align columns with training feature space
# --------------------------------------------------
X_test_tmp_extra = X_test_tmp_extra.reindex(
    columns=train_feature_cols_extra,
    fill_value=0
)

print("Test set ready.")
print("Shape:", X_test_tmp_extra.shape)
print("NaNs:", X_test_tmp_extra.isna().sum().sum())


## **9.2 Final RandomForest Training (Blend Partner)**

In [ ]:

print("Training RandomForest for blending...")

final_rf = RandomForestRegressor(
    random_state=RANDOM_SEED,
    n_jobs=-1,
    n_estimators=600,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt"
)

final_rf.fit(X_full_processed_extra, y)

print("RandomForest ready for blending.")


## **9.2 Final Histogram Gradient Boosting Training (Blend Partner)**

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

print("Training HistGradientBoosting for blending...")

final_hgb = HistGradientBoostingRegressor(
    random_state=RANDOM_SEED,
    max_iter=300,
    learning_rate=0.1,
    max_depth=None,
    min_samples_leaf=50
)

final_hgb.fit(X_full_processed_extra, y)

print("HistGradientBoosting ready for blending.")


## **9.3 Blend Predictions (Simple Average)**

In [ ]:
## **9.3 Blend Predictions and Submission (Simple Average)**

"""
README
------
Generate blended predictions (ExtraTrees + RandomForest)
using simple averaging and export a Kaggle-compliant
submission file.

Expected format:
- carID
- price
"""

# --------------------------------------------------
# Sanity checks
# --------------------------------------------------
assert "final_extra" in globals(), "Final ExtraTrees model not found"
assert "final_rf" in globals(), "Final RandomForest model not found"
assert "X_test_tmp_extra" in globals(), "Processed test matrix not found"
assert "test_ids" in globals(), "test_ids not found"

print("Generating blended predictions...")

# --------------------------------------------------
# Individual predictions
# --------------------------------------------------
y_pred_extra = final_extra.predict(X_test_tmp_extra)
y_pred_rf = final_rf.predict(X_test_tmp_extra)

# --------------------------------------------------
# Simple average blend
# --------------------------------------------------
y_pred_blend = 0.5 * y_pred_extra + 0.5 * y_pred_rf

print("Blending completed.")
print("Sample predictions:", y_pred_blend[:5])
print("Min / Max:", float(y_pred_blend.min()), float(y_pred_blend.max()))

# --------------------------------------------------
# Build submission dataframe (Kaggle-safe)
# --------------------------------------------------
submission = pd.DataFrame({
    "carID": test_ids,
    "price": y_pred_blend
})

# --------------------------------------------------
# Final validation
# --------------------------------------------------
assert submission.columns.tolist() == ["carID", "price"]
assert len(submission) == len(test_ids)

print("Submission preview:")
print(submission.head())
print("Submission shape:", submission.shape)

# --------------------------------------------------
# Save to CSV
# --------------------------------------------------
submission_path = "submission_blend_extratrees_rf.csv"
submission.to_csv(submission_path, index=False)

print(f"Submission file saved: {submission_path}")




## **9.4 Blend Predictions (Weighted Average)**

In [ ]:
## **9.4 Weighted Blend Predictions and Submission**

"""
README
------
Generate weighted blended predictions (ExtraTrees + RandomForest)
and export a Kaggle-compliant submission file.
"""

# --------------------------------------------------
# Sanity checks (hard fail)
# --------------------------------------------------
assert "final_extra" in globals(), "final_extra not found"
assert "final_rf" in globals(), "final_rf not found"
assert "X_test_tmp_extra" in globals(), "Processed test matrix not found"
assert "test_ids" in globals(), "test_ids not found"

# --------------------------------------------------
# Blend weights (5% ET + 95% RF)
# --------------------------------------------------
w_extra, w_rf = 0.05, 0.95
assert abs(w_extra + w_rf - 1.0) < 1e-9

print("Generating weighted blend predictions...")

# --------------------------------------------------
# Predictions
# --------------------------------------------------
y_test_extra = final_extra.predict(X_test_tmp_extra)
y_test_rf = final_rf.predict(X_test_tmp_extra)

y_test_blend = w_extra * y_test_extra + w_rf * y_test_rf

print("Done.")
print("n_preds:", len(y_test_blend))
print("min/max:", float(y_test_blend.min()), float(y_test_blend.max()))

# --------------------------------------------------
# Build submission (Kaggle-safe)
# --------------------------------------------------
assert len(y_test_blend) == len(test_ids), "Prediction / ID length mismatch"

submission = pd.DataFrame({
    "carID": test_ids,
    "price": y_test_blend
})

# Final checks
assert submission.columns.tolist() == ["carID", "price"]

print("Submission preview:")
print(submission.head())

# --------------------------------------------------
# Save
# --------------------------------------------------
out_path = "submission_blend_05ET_95RF.csv"
submission.to_csv(out_path, index=False)

print("Saved:", out_path)





## **9.5 Blend Predictions ET+RF+HGB**

In [ ]:
"""
README
------
Generate blended predictions using:
- ExtraTrees
- RandomForest
- HistGradientBoosting

Strategy:
- Simple average of three strong but diverse models
"""

# --------------------------------------------------
# Sanity checks
# --------------------------------------------------
assert "final_extra" in globals()
assert "final_rf" in globals()
assert "final_hgb" in globals()
assert "X_test_tmp_extra" in globals()
assert "test_ids" in globals()

print("Generating blended predictions (ET + RF + HGB)...")

# --------------------------------------------------
# Individual predictions
# --------------------------------------------------
y_et = final_extra.predict(X_test_tmp_extra)
y_rf = final_rf.predict(X_test_tmp_extra)
y_hgb = final_hgb.predict(X_test_tmp_extra)

# --------------------------------------------------
# Simple average blend (3 models)
# --------------------------------------------------
y_pred_blend = (y_et + y_rf + y_hgb) / 3.0

print("Blending completed.")
print("Sample predictions:", y_pred_blend[:5])
print("Min / Max:", float(y_pred_blend.min()), float(y_pred_blend.max()))

# --------------------------------------------------
# Submission
# --------------------------------------------------
submission = pd.DataFrame({
    "carID": test_ids,
    "price": y_pred_blend
})

submission_path = "submission_blend_ET_RF_HGB.csv"
submission.to_csv(submission_path, index=False)

print("Submission saved:", submission_path)


## **9.6 Blend Predictions 0.2ET+0.4RF+0.4HGB**

In [ ]:
"""
README
------
Weighted blend:
- ExtraTrees: 0.2
- RandomForest: 0.4
- HistGradientBoosting: 0.4
"""

# --------------------------------------------------
# Weights
# --------------------------------------------------
w_et, w_rf, w_hgb = 0.2, 0.4, 0.4
assert abs(w_et + w_rf + w_hgb - 1.0) < 1e-9

print("Generating weighted blended predictions...")

y_pred_weighted = (
    w_et * y_et +
    w_rf * y_rf +
    w_hgb * y_hgb
)

# --------------------------------------------------
# Submission
# --------------------------------------------------
submission = pd.DataFrame({
    "carID": test_ids,
    "price": y_pred_weighted
})

out_path = "submission_weighted_ET_RF_HGB.csv"
submission.to_csv(out_path, index=False)

print("Weighted submission saved:", out_path)


# **10 ANALYTICS INTERFACE FOR NEW-DATA INFERENCE**


In [ ]:
# ==========================================================
# 10.0 — INFERENCE SANITY CHECK (PIPELINE ORIGINAL)
# ==========================================================
"""
README
------
Run this before any UI.

This cell verifies that the FINAL trained objects and preprocessing artifacts
exist in memory (produced by the pipeline) and are ready for inference.

If this cell fails, you must run the missing upstream section(s).
"""

required = [
    "final_extra",
    "final_rf",
    "final_hgb",
    "final_ohe",
    "artifacts_extra",
    "train_feature_cols_extra",
    "int_cols",
    "float_cols",
    "cat_cols",
    "valid_models",
    "clean_df",
    "preprocess_for_search",
]

missing = [name for name in required if name not in globals()]
assert not missing, f"Missing in globals(): {missing}"

print("All required objects found ✓  Ready for dashboard in the same notebook.")


## **10.1 Single car Price Prediction Function**

In [ ]:
# ==========================================================
# 10.1  SINGLE-CAR PREDICTION (PIPELINE ORIGINAL)
# ==========================================================
"""
README
------
Canonical inference function for the Cars4You project.

- Uses the SAME cleaning function: clean_df()
- Uses the SAME preprocessing: preprocess_for_search(fit=False)
- Reuses trained artifacts: final_ohe + artifacts_extra["fill_values"]
- Aligns feature space with training: train_feature_cols_extra
- Produces:
  - prediction per model (ET / RF / HGB)
  - weighted ensemble (ET 0.2 / RF 0.4 / HGB 0.4)
  - a model-disagreement indicator (std dev)
"""

import numpy as np
import pandas as pd

W_ET, W_RF, W_HGB = 0.2, 0.4, 0.4
assert abs(W_ET + W_RF + W_HGB - 1.0) < 1e-9

def predict_single_car_pipeline(
    brand,
    model,
    year,
    mileage,
    fuelType,
    transmission,
    engineSize,
    tax=None,
    mpg=None,
    paintQuality=None,
    previousOwners=None
):
    # --------------------------------------------------
    # 1) Build single-row DF (match training schema)
    # --------------------------------------------------
    input_df = pd.DataFrame([{
        "carID": -1,                 # technical id
        "Brand": brand,
        "model": model,
        "year": int(year),
        "mileage": mileage,
        "fuelType": fuelType,
        "transmission": transmission,
        "engineSize": engineSize,
        "tax": tax,
        "mpg": mpg,
        "paintQuality%": paintQuality,
        "previousOwners": previousOwners,
        "hasDamage": 0               # neutral placeholder if pipeline expects it
    }])

    # --------------------------------------------------
    # 2) Deterministic cleaning (same as pipeline)
    # --------------------------------------------------
    input_clean = clean_df(input_df, valid_models)

    # --------------------------------------------------
    # 3) Preprocess (NO FIT) using trained artifacts
    # --------------------------------------------------
    X_proc, _ = preprocess_for_search(
        input_clean,
        int_cols=int_cols,
        float_cols=float_cols,
        cat_cols=cat_cols,
        ohe=final_ohe,
        apply_outliers=False,
        fit=False,
        fill_values=artifacts_extra["fill_values"],
        outlier_info=artifacts_extra.get("outlier_info")
    )

    # --------------------------------------------------
    # 4) Align features to training space
    # --------------------------------------------------
    X_proc = X_proc.reindex(
        columns=train_feature_cols_extra,
        fill_value=0
    )

    # --------------------------------------------------
    # 5) Predictions
    # --------------------------------------------------
    y_et  = float(final_extra.predict(X_proc)[0])
    y_rf  = float(final_rf.predict(X_proc)[0])
    y_hgb = float(final_hgb.predict(X_proc)[0])

    y_ens = W_ET*y_et + W_RF*y_rf + W_HGB*y_hgb
    std  = float(np.std([y_et, y_rf, y_hgb]))

    return {
        "ExtraTrees": y_et,
        "RandomForest": y_rf,
        "HistGradientBoosting": y_hgb,
        "Ensemble(0.2/0.4/0.4)": y_ens,
        "DisagreementStd": std
    }


# **10.2 Interactive Analytics Interface (Individual Car Prediction)**

In [ ]:
# ==========================================================
# 10.2 — INTERACTIVE DASHBOARD (ORIGINAL PIPELINE)
# ==========================================================
"""
README
------
Gradio front-end for single-car price prediction using
the ORIGINAL Cars4You pipeline.

Features:
- Brand dropdown
- Model dropdown (dependent on Brand)
- Correct transmission dropdown
- Calls predict_single_car_pipeline()
- Displays individual models + ensemble + chart
"""

import gradio as gr
import plotly.graph_objects as go

# --------------------------------------------------
# Helpers for dropdown logic
# --------------------------------------------------
brand_list = sorted(valid_models.keys())

def update_models(brand):
    models = sorted(valid_models.get(brand, []))
    return gr.Dropdown(
        choices=models,
        value=models[0] if models else None
    )

# --------------------------------------------------
# UI
# --------------------------------------------------
with gr.Blocks(title="Cars4You — Car Price Prediction Dashboard (Original Pipeline)") as demo:

    gr.Markdown("## 🚗 Cars4You — Car Price Prediction Dashboard")

    # ------------------------------
    # Inputs
    # ------------------------------
    with gr.Row():
        brand = gr.Dropdown(
            choices=brand_list,
            value=brand_list[0],
            label="Brand"
        )
        model = gr.Dropdown(
            choices=sorted(valid_models[brand_list[0]]),
            value=sorted(valid_models[brand_list[0]])[0],
            label="Model"
        )

    with gr.Row():
        year = gr.Slider(1990, 2024, step=1, value=2018, label="Year")
        mileage = gr.Number(value=60000, label="Mileage (km)")
        engine = gr.Number(value=2.0, label="Engine size (L)")

    with gr.Row():
        fuel = gr.Dropdown(
            ["PETROL", "DIESEL", "HYBRID", "ELECTRIC"],
            value="DIESEL",
            label="Fuel type"
        )
        transmission = gr.Dropdown(
            ["MANUAL", "AUTOMATIC"],
            value="MANUAL",
            label="Transmission"
        )

    with gr.Row():
        tax = gr.Number(value=0, label="Tax (optional)")
        mpg = gr.Number(value=0, label="MPG (optional)")
        paint_q = gr.Number(value=0, label="Paint quality % (optional)")
        prev_owners = gr.Number(value=1, label="Previous owners (optional)")

    predict_btn = gr.Button("Predict", variant="primary")

    # ------------------------------
    # Outputs
    # ------------------------------
    gr.Markdown("### Outputs")

    with gr.Row():
        out_et = gr.Number(label="ExtraTrees (€)")
        out_rf = gr.Number(label="RandomForest (€)")
        out_hgb = gr.Number(label="HGB (€)")
        out_ens = gr.Number(label="Ensemble (€)")
        out_std = gr.Number(label="Model disagreement (std)")

    plot_out = gr.Plot()

    # ------------------------------
    # Wiring
    # ------------------------------
    brand.change(
        fn=update_models,
        inputs=brand,
        outputs=model
    )

    def ui_predict(*args):
        res = predict_single_car_pipeline(*args)

        fig = go.Figure(
            data=[
                go.Bar(
                    x=["ExtraTrees", "RandomForest", "HGB", "Ensemble"],
                    y=[
                        res["ExtraTrees"],
                        res["RandomForest"],
                        res["HistGradientBoosting"],
                        res["Ensemble(0.2/0.4/0.4)"]
                    ]
                )
            ]
        )
        fig.update_layout(
            title="Cars4You — Price Predictions (€)",
            yaxis_title="Predicted price",
            showlegend=False
        )

        return (
            round(res["ExtraTrees"], 0),
            round(res["RandomForest"], 0),
            round(res["HistGradientBoosting"], 0),
            round(res["Ensemble(0.2/0.4/0.4)"], 0),
            round(res["DisagreementStd"], 0),
            fig
        )

    predict_btn.click(
        fn=ui_predict,
        inputs=[
            brand, model, year, mileage, fuel,
            transmission, engine, tax, mpg,
            paint_q, prev_owners
        ],
        outputs=[
            out_et, out_rf, out_hgb,
            out_ens, out_std, plot_out
        ]
    )

demo.launch(debug=True)


